# SARIMA MODEL: NBA API

`Authors:` Sarah Beltran, Sofia Maldonado & Aissa 

`Date:` 17/02/2026

---

In [ ]:
# %pip install nbformat nba_api

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp311-cp311-win_amd64.whl.metadata (4.2 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
Using cached rpds_py-0.30.0-cp311-cp311-win_amd64.whl (236 kB)

   ---------------------------------------- 0/7 [fastjsonschema]
   ---------------------------------------- 0/7 [fastjsonschema]
   ---------------------------------------- 0/7 [fastjsonschema]
   -----------------------

In [10]:
# libraries 
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd
import plotly.graph_objects as go
import nbformat
from statsmodels.tsa.stattools import adfuller, acf, pacf
from plotly.subplots import make_subplots
import numpy as np


In [11]:
games = leaguegamefinder.LeagueGameFinder(
    season_type_nullable='Regular Season'
).get_data_frames()[0]
df_games = pd.DataFrame(games)

# converts date
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])

# regular season 2024-2025
# SEASON_ID: '22024'
df_games = df_games[df_games['SEASON_ID'] == '22024']

# no play-in/offs
df_clean = df_games[df_games['GAME_DATE'] <= '2025-02-15'].copy()

# ended games
df_clean = df_clean[df_clean['WL'].notna()]

# total points per game
df_clean['total_points'] = df_clean['PTS']

# daily time series points per day
ts_nba = (
    df_clean
    .groupby('GAME_DATE')['total_points']
    .sum()
    .asfreq('D')
    .fillna(0)
)

In [12]:
# original time series graph
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ts_nba.index,
        y=ts_nba.values,
        mode='lines',
        name='Puntos Diarios'
    )
)

fig.update_layout(
    title='Volumen Diario de Puntos en la NBA (Regular Season)',
    xaxis_title='Fecha',
    yaxis_title='Total de Puntos'
)

fig.show()

In [14]:
# stationariry tests

def check_stationarity(series, title="series"):
    result = adfuller(series.dropna())
    print(f'ADF test: {title}')
    print(f'statistics ADF: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    is_stationary = result[1] < 0.05
    print(f"Is it stationary? {'YES' if is_stationary else 'NO'}\n")
    return is_stationary

# 1. original
check_stationarity(ts_nba, "original level (NBA)")

# 2. first diff
ts_nba_diff = ts_nba.diff()

# 3. diff series
check_stationarity(ts_nba_diff, "first differentiation (d=1)")

# comparative subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "original series (NBA – not stationary)",
        "differentiated series (NBA – stationary d=1)"
    )
)

# Serie original
fig.add_trace(
    go.Scatter(x=ts_nba.index, y=ts_nba, name='original'),
    row=1, col=1
)

# Serie diferenciada
fig.add_trace(
    go.Scatter(x=ts_nba_diff.index, y=ts_nba_diff, name='differentiated'),
    row=1, col=2
)

fig.update_layout(
    title_text="comparative differentiation effect (NBA)",
    showlegend=False,
    height=500
)

fig.show()


ADF test: original level (NBA)
statistics ADF: -2.3656
p-value: 0.1517
Is it stationary? NO

ADF test: first differentiation (d=1)
statistics ADF: -4.4481
p-value: 0.0002
Is it stationary? YES



In [ ]:
# using diff series cause d=1
ts_analysis = ts_nba.diff().dropna()

lags = 30      # 30 days
alpha = 0.05  

acf_vals = acf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]
pacf_vals = pacf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]

n = len(ts_analysis)
conf_interval = 1.96 / np.sqrt(n)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "autocorrelation function (ACF) – determine MA(q)",
        "partial autocorrelation (PACF) – determine AR(p)"
    ),
    vertical_spacing=0.15
)

# ACF
fig.add_trace(
    go.Bar(
        x=list(range(1, lags + 1)),
        y=acf_vals,
        name='ACF',
        showlegend=False
    ),
    row=1, col=1
)

fig.add_shape(
    type="rect",
    x0=0.5, y0=-conf_interval, x1=lags + 0.5, y1=conf_interval,
    line=dict(width=0),
    fillcolor="rgba(0,0,0,0.1)",
    row=1, col=1
)

fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=1, col=1)

# PACF
fig.add_trace(
    go.Bar(
        x=list(range(1, lags + 1)),
        y=pacf_vals,
        name='PACF',
        showlegend=False
    ),
    row=2, col=1
)

fig.add_shape(
    type="rect",
    x0=0.5, y0=-conf_interval, x1=lags + 0.5, y1=conf_interval,
    line=dict(width=0),
    fillcolor="rgba(0,0,0,0.1)",
    row=2, col=1
)

fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=2, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=2, col=1)

fig.update_layout(
    title="<b>structure diagnosis: ACF y PACF (NBA)</b><br><sup>diff. series</sup>",
    template="plotly_white",
    height=700,
    bargap=0.8
)

# lags (red lines)
# for i in [7, 14, 21, 28]:
#     fig.add_vline(x=i, line_width=1, line_dash="dot", line_color="red", opacity=0.5)

fig.show()